# freeze-requires-grad — faded example 1: Freeze Backbone: Collect Only Trainable Head Params

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `freeze-requires-grad`. Running the beacon reports progress on the `PyTorch: freeze via requires_grad=False` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: freeze via requires_grad=False` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`freeze-requires-grad`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "freeze-requires-grad"
DD_SUBTOPIC = "PyTorch: freeze via requires_grad=False"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Setting `requires_grad = False` on all backbone parameters before attaching a new head is the core transfer-learning idiom. After the head replacement, only the new head's parameters are trainable. Collecting `[p for p in model.parameters() if p.requires_grad]` gives exactly the optimizer input for fine-tuning.

## Faded exercise 1

Implement `freeze_and_collect(model, new_out_features)` that: (1) sets `requires_grad = False` on every parameter currently in `model`; (2) replaces `model.fc` with a new `nn.Linear(model.fc.in_features, new_out_features)`; (3) returns `[p for p in model.parameters() if p.requires_grad]`.

**Fill in:** Iterate over model.parameters() and set p.requires_grad = False for each, then replace model.fc with nn.Linear(model.fc.in_features, new_out_features).

In [ ]:
import torch as t
import torch.nn as nn

def freeze_and_collect(model: nn.Module, new_out_features: int):
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, new_out_features)
    return [p for p in model.parameters() if p.requires_grad]


def _test():
    import torch as t
    import torch.nn as nn
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.Linear(10, 20)
            self.fc  = nn.Linear(20, 100)
        def forward(self, x):
            return self.fc(self.enc(x))
    model = Net()
    trainable = freeze_and_collect(model, 5)
    # enc should be frozen
    assert not model.enc.weight.requires_grad
    # new head should be trainable
    assert model.fc.weight.requires_grad
    assert model.fc.out_features == 5
    # trainable list has exactly 2 tensors
    assert len(trainable) == 2


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def freeze_and_collect(model: nn.Module, new_out_features: int):
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, new_out_features)
    return [p for p in model.parameters() if p.requires_grad]
```
</details>